In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
import numpy as np

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="INDICADORES_ESSALUD"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2= engine2.connect()

from datetime import datetime
from sqlalchemy import text

In [2]:
import numpy as np
import oracledb
import pandas as pd
from sqlalchemy import create_engine, text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb

# Configuración de la conexión
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname = config("HOST4_BDI_POSTGRES")
service_name = "WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@', connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

# Tamaño de los bloques
chunk_size = 350000
offset = 0

# Cargar datos de referencia desde PostgreSQL
oricas = pd.read_sql_query("SELECT id_oricas, ori_cod FROM dim_oricas", con=engine2)
cas = pd.read_sql_query(f"SELECT id_cas,cod_cas, cod_red FROM  dim_cas where cod_cas is not null", con=connection2)
red = pd.read_sql_query("SELECT id_red, cod_red FROM dim_red", con=engine2)
tippac = pd.read_sql_query("SELECT id_tippac, cod_tpac FROM dim_tippac", con=engine2)
estreg = pd.read_sql_query("SELECT id_estreg, cod_est FROM dim_estreg", con=engine2)

while True:
    query = f"""
    SELECT *
    FROM (
        SELECT 
            oricenasicod,
            cenasicod,
            pacsecnum,
            tipopacicod,
            pacfilfec,
            pachispasfec,
            pachisdepfec,
            pacfallfec,
            pacestregcod,
            paccrefec,
            pacmodfec,
            ROW_NUMBER() OVER (ORDER BY pacsecnum) AS rn
        FROM cmpac10
    )
    WHERE rn > {offset} AND rn <= {offset + chunk_size}
    """

    chunk = pd.read_sql_query(query, con=engine0)
    if chunk.empty:
        break

    chunk = chunk.rename(columns={
        'oricenasicod': 'ori_cod',
        'cenasicod': 'cod_cas',
        'pacsecnum': 'pac_sec',
        'tipopacicod': 'cod_tpac',
        'pacfilfec': 'fec_fil',
        'pacfallfec': 'fec_fall',
        'pacestregcod': 'cod_est',
        'paccrefec': 'fec_cre',
        'pacmodfec': 'fec_mod'
    })

    # Transformaciones y combinaciones con otros DataFrames
    oricas['ori_cod'] = oricas['ori_cod'].str.strip()
    chunk['ori_cod'] = chunk['ori_cod'].str.strip()
    chunk = pd.merge(chunk, oricas, on='ori_cod', how="left").drop('ori_cod', axis=1)
    cas['cod_cas'] = cas['cod_cas'].str.strip()
    cas['cod_red'] = cas['cod_red'].str.strip()
    chunk['cod_cas'] = chunk['cod_cas'].str.strip()
    chunk = pd.merge(chunk, cas,how='left',on='cod_cas').drop('cod_cas', axis=1)
    red['cod_red'] = red['cod_red'].str.strip()
    chunk['cod_red'] = chunk['cod_red'].str.strip()
    chunk = pd.merge(chunk, red,how='left',on='cod_red').drop('cod_red', axis=1)
    tippac['cod_tpac'] = tippac['cod_tpac'].str.strip()
    chunk['cod_tpac'] = chunk['cod_tpac'].str.strip()
    chunk = pd.merge(chunk, tippac, on='cod_tpac', how="left").drop('cod_tpac', axis=1)
    estreg['cod_est'] = estreg['cod_est'].str.strip()
    chunk['cod_est'] = chunk['cod_est'].str.strip()
    chunk = pd.merge(chunk, estreg, on='cod_est', how="left").drop('cod_est', axis=1)

    # Convertir columnas de fecha a datetime y manejar NaT
    date_columns = ['fec_fil', 'fec_fall', 'fec_cre', 'fec_mod']

    # Convertir las columnas de fecha a datetime, reemplazando errores con NaT
    for col in date_columns:
        chunk[col] = pd.to_datetime(chunk[col], errors='coerce')

    # Reemplazar NaT por None en columnas de fecha
    for col in date_columns:
        chunk[col] = chunk[col].apply(lambda x: None if pd.isna(x) else x)

    # Convertir el resto de NaN a None
    chunk = chunk.applymap(lambda x: None if pd.isna(x) else x)

    print('subiendo bloques')
    # Upsert en PostgreSQL
    for _, row in chunk.iterrows():
        row_dict = row.to_dict()

        # Verificar y convertir cualquier valor de cadena 'NaT' o pd.NaT a None
        for key, value in row_dict.items():
            if value == 'NaT' or value is pd.NaT or str(value) == 'NaT':
                row_dict[key] = None

        sql = text("""
        INSERT INTO dim_pac (pac_sec, id_oricas, id_red, id_cas, id_tippac, fec_fil, fec_fall, id_estreg, fec_cre, fec_mod)
        VALUES (:pac_sec, :id_oricas, :id_red, :id_cas, :id_tippac, :fec_fil, :fec_fall, :id_estreg, :fec_cre, :fec_mod)
        """)
        
        engine2.execute(sql, **row_dict)
        
    print(f'bloque {_} subido')
    offset += chunk_size


subiendo bloques
bloque 351403 subido
subiendo bloques
bloque 352029 subido
subiendo bloques
bloque 352323 subido
subiendo bloques


OperationalError: (psycopg2.OperationalError) server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.

(Background on this error at: https://sqlalche.me/e/14/e3q8)

In [ ]:
connection2.close()
connection0.close()
engine2.dispose()
engine0.dispose()